# AI-Based Weather Condition Classification System

Step-by-step notebook version: data loading, model building, training, evaluation, and single-image prediction.

Make sure you've run `python utils/prepare_dataset.py` first so that `data/dataset/{train,val,test}` exist.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import config
from data_loader import get_data_generators
from model import build_model
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

## 1. Load data generators

In [ ]:
train_gen, val_gen, test_gen = get_data_generators()
print(train_gen.class_indices)

## 2. Peek at a batch of images

In [ ]:
images, labels = next(train_gen)
class_names = list(train_gen.class_indices.keys())

plt.figure(figsize=(10, 8))
for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[labels[i].argmax()])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Build the model (transfer learning, head-only phase)

In [ ]:
model, base_model = build_model(fine_tune=False)
model.compile(
    optimizer=Adam(learning_rate=config.HEAD_LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 4. Train (short demo run — reduce epochs here; use src/train.py for the full two-phase pipeline)

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5
)

## 5. Plot accuracy / loss curves

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend(); plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend(); plt.title('Loss')
plt.show()

## 6. Evaluate on test set

In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print(f'Test accuracy: {test_acc*100:.2f}%')

## 7. Save the model

In [ ]:
os.makedirs(config.MODELS_DIR, exist_ok=True)
model.save(config.FINAL_MODEL_PATH)
print('Saved to', config.FINAL_MODEL_PATH)